# Silver 0: split the long PDFs into parts

Task `silver_split` of the job `ingest_financebench`. Cuts every PDF in `bronze_pdfs` that has more than `part_pages` pages into **parts** of at most `part_pages` pages, and stores the bytes of each part in `silver_parts`. A PDF with `part_pages` pages or fewer is one part with its original bytes.

**Why.** The parser throttles (`Request rate limit exceeded`) when it gets a long PDF in one request, and a failed page throws away the whole parse. In a test, a 155-page filing lost 27 pages when parsed whole, and came back complete in four parts of 50, 50, 50 and 5 pages. A part that fails is retried alone, so a retry costs at most `part_pages` pages. The parser also rejects PDFs over 500 pages: as parts they can be parsed.

| Rule | Why |
| --- | --- |
| A PDF already in `silver_parsed` stays one part, stored **without bytes** | It is parsed, so nothing reads its bytes again, and the table does not copy the PDF |
| Parts are stored in a table, not as files | Nothing to track in the volume, and a part keeps its original `path` and `page_offset` |
| A PDF that cannot be read here is one part with its original bytes | The parser decides, and a failure is logged like any other |
| Only PDFs without parts are cut, one PDF at a time | Running it twice does nothing the second time. Reading all the PDFs at once through the driver failed (`UnknownException`) on the real data |

The real page number of a page is `page_offset + page_id + 1`, so the pages, the log and the documents keep the page numbers of the original PDF. The cutting logic is in [splitting.py](./splitting.py) (tested with `pytest`).

In [ ]:
%pip install pypdf -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
from pyspark.sql import functions as F

from splitting import split_pdf
from tables import Names, create_tables

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("part_pages", "50")  # PDFs with more pages are cut into parts of this size

n = Names(dbutils.widgets.get("catalog"), dbutils.widgets.get("schema"))
part_pages = int(dbutils.widgets.get("part_pages"))

create_tables(spark, n)

In [ ]:
# PDFs that are parsed already stay whole. Nothing reads their bytes again, so the part has none.
spark.sql(f"""
INSERT INTO {n.parts}
SELECT path, 1, 0, max(pdf_pages), CAST(NULL AS BINARY), current_timestamp()
FROM {n.parsed} WHERE path NOT IN (SELECT path FROM {n.parts}) GROUP BY path
""")

# The other PDFs are cut one at a time: a PDF is read, cut and stored before the next one is read.
todo = [r.path for r in spark.table(n.bronze).select("path").join(spark.table(n.parts).select("path").distinct(), "path", "left_anti").collect()]
print(f"{len(todo)} PDFs to split.")

cut = 0
for path in todo:
    content = spark.table(n.bronze).filter(F.col("path") == path).select("content").first().content
    parts = split_pdf(bytes(content), part_pages)
    rows = [(path, part_no, offset, pages, content) for part_no, offset, pages, content in parts]
    (
        spark.createDataFrame(rows, "path string, part_no int, page_offset int, pages int, content binary")
        .withColumn("split_at", F.current_timestamp())
        .repartition(1)  # one file per PDF: the parse stream takes a few files per batch
        .write.mode("append").saveAsTable(n.parts)
    )
    cut += len(parts) > 1
    print(f"{path.split('/')[-1]}: {len(parts)} part(s)")
print(f"{cut} PDFs cut into parts.")

In [ ]:
display(spark.sql(f"SELECT count(DISTINCT path) AS pdfs, count(*) AS parts, max(pages) AS longest_part FROM {n.parts}"))